In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [ ]:
RAW_DIR = "data_engine1/raw"

patients = pd.read_csv(f"{RAW_DIR}/patients.csv")
conditions = pd.read_csv(f"{RAW_DIR}/conditions.csv")
observations = pd.read_csv(f"{RAW_DIR}/observations.csv")

print(f"Total Patients loaded: {len(patients)}")
print(f"Total Conditions loaded: {len(conditions)}")
print(f"Total Observations loaded: {len(observations)}")

patients[['Id', 'BIRTHDATE', 'GENDER', 'RACE']].head(3)

Total Patients loaded: 113
Total Conditions loaded: 4748
Total Observations loaded: 114342


,Id,BIRTHDATE,GENDER,RACE
0,fe621c76-a591-b7be-5668-b77f00240d82,2001-12-29,F,white
1,5f01f823-c8e8-9e40-c911-54b7b1ad843a,1969-07-11,F,white
2,cba958fc-9355-f095-bf77-d56a20511339,1997-03-28,M,white


In [6]:
#Patients diagnosed with major chronic conditions
high_risk_keywords = ['Diabetes', 'Hypertension', 'Coronary', 'Metabolic syndrome']
pattern = '|'.join(high_risk_keywords)

risk_conditions = conditions[conditions['DESCRIPTION'].str.contains(pattern, case=False, na=False)]
high_risk_ids = set(risk_conditions['PATIENT'].unique())

# Process demographics & apply target flag
patients['BIRTHDATE'] = pd.to_datetime(patients['BIRTHDATE'], errors='coerce')
patients['age'] = datetime.now().year - patients['BIRTHDATE'].dt.year

demographics = patients[['Id', 'GENDER', 'RACE', 'age']].rename(columns={'Id': 'PATIENT'})
demographics['is_high_risk'] = demographics['PATIENT'].apply(lambda x: 1 if x in high_risk_ids else 0)

# Check class balance
print("--- Class Distribution ---")
print(demographics['is_high_risk'].value_counts(normalize=True) * 100)

--- Class Distribution ---
is_high_risk
0    53.097345
1    46.902655
Name: proportion, dtype: float64


In [7]:
observations['DATE'] = pd.to_datetime(observations['DATE'], errors='coerce')

# Grab the latest observation per patient per test type
latest_obs = observations.sort_values('DATE', ascending=False).drop_duplicates(
    subset=['PATIENT', 'DESCRIPTION']
)

core_vitals = [
    'Body Mass Index',
    'Systolic Blood Pressure',
    'Diastolic Blood Pressure',
    'Glucose',
    'Total Cholesterol'
]
filtered_obs = latest_obs[latest_obs['DESCRIPTION'].isin(core_vitals)]

# Pivot rows into columns!
lab_matrix = filtered_obs.pivot(index='PATIENT', columns='DESCRIPTION', values='VALUE').reset_index()

# Ensure numeric types
for col in core_vitals:
    if col in lab_matrix.columns:
        lab_matrix[col] = pd.to_numeric(lab_matrix[col], errors='coerce')

# Inspect our pivoted labs
lab_matrix.head(3)

DESCRIPTION,PATIENT,Diastolic Blood Pressure,Systolic Blood Pressure
0,01ae1af3-937a-83eb-2998-8cb6efc21b79,83.0,99.0
1,02bfaacd-a829-fab9-1e7d-7d0fc44b1b69,77.0,132.0
2,03f66b92-3aca-380b-e31c-492cbb90c056,80.0,120.0


In [ ]:
master_df = pd.merge(demographics, lab_matrix, on='PATIENT', how='left').drop(columns=['PATIENT'])

# Save our matrix for DVC tracking later
os.makedirs("data_engine1/processed", exist_ok=True)
master_df.to_csv("data_engine1/processed/synthea_ml_matrix.csv", index=False)

print(f"Master Matrix Shape: {master_df.shape}")
print("\n--- Missing Value Count Per Column ---")
print(master_df.isnull().sum())

# Display the final ML-ready table
master_df.head()

Master Matrix Shape: (113, 6)

--- Missing Value Count Per Column ---
GENDER                      0
RACE                        0
age                         0
is_high_risk                0
Diastolic Blood Pressure    0
Systolic Blood Pressure     0
dtype: int64


,GENDER,RACE,age,is_high_risk,Diastolic Blood Pressure,Systolic Blood Pressure
0,F,white,25,0,70.0,126.0
1,F,white,57,0,80.0,105.0
2,M,white,29,0,76.0,119.0
3,F,white,53,1,109.0,143.0
4,M,asian,27,0,76.0,109.0
